In [6]:
%run Table

StatementMeta(, e58fdf1e-0c00-456c-8c2b-2498867b5cfd, 65, Finished, Available, Finished, True)

In [2]:
%run ClassManagement

StatementMeta(, e58fdf1e-0c00-456c-8c2b-2498867b5cfd, 33, Finished, Available, Finished, True)

In [3]:
import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants

StatementMeta(, e58fdf1e-0c00-456c-8c2b-2498867b5cfd, 34, Finished, Available, Finished, False)

In [21]:
class FactTaxiDailyTransformer:

    def __init__(self):
        self.sources = ClassManagement.get_all_subclass_instances(TaxiSilverTable)
        self.factTaxiDaily = FactTaxiDaily()

    def get_new_data(self, source):
        new_months = self.factTaxiDaily.new_months(source, source.TAXI_TYPE_ID).df
        return source.enrich_by_year_month()\
                    .filter_by_year_months(new_months)\
                    .drop_year_month()\
                    .enrich_by_value_columns()\
                    .enrich_by_taxi_type_id()\
                    .enrich_by_date()\
                    .df

    def aggregate(self, df):
        return df.groupBy(*self.factTaxiDaily.PKS)\
            .agg(F.count("*").alias("trip_count"),
                F.sum("trip_distance").alias("total_distance"),
                F.avg("trip_distance").alias("avg_trip_distance"),
                F.sum("fare_amount").alias("total_fare"),
                F.avg("fare_amount").alias("avg_fare"),
                F.sum("total_amount").alias("total_amount"),
                F.avg("total_amount").alias("avg_total_amount"),
                F.sum("tip_amount").alias("total_tip"),
                F.avg("tip_amount").alias("avg_tip"))
    
    def cast_columns(self, df):
        return df.withColumn("pickup_location_id", F.col("pickup_location_id").cast("int"))\
            .withColumn("dropoff_location_id", F.col("dropoff_location_id").cast("int"))

    def transform(self):
        for source in self.sources:
            new_data = self.get_new_data(source)
            aggregated_data = self.aggregate(new_data)
            aggregated_data = self.cast_columns(aggregated_data)
            if not aggregated_data.isEmpty():
                FactTaxiDaily(aggregated_data).append()
            

StatementMeta(, e58fdf1e-0c00-456c-8c2b-2498867b5cfd, 107, Finished, Available, Finished, False)

In [22]:
FactTaxiDailyTransformer().transform()

StatementMeta(, e58fdf1e-0c00-456c-8c2b-2498867b5cfd, 108, Submitted, Running, Running, True)